# 01 · xAPI Ingestion — Bronze → Silver → Gold

**Sprint 1, Task 1.3–1.4**  
Reads raw JSON files uploaded to `Files/raw/` by `02_upload_to_fabric.py` and creates
medallion Delta Lake tables in the default Lakehouse.

| Layer  | Tables created |
|--------|----------------|
| Bronze | `bronze_xapi_events`, `dim_courses`, `dim_learners` |
| Silver | `silver_xapi_events` |
| Gold   | `gold_course_completion`, `gold_module_dropoff`, `gold_department_skills`, `gold_engagement_hourly` |

> **Run order:** Execute cells top-to-bottom. Each section is idempotent (`overwrite` / `CREATE OR REPLACE`).

## 0 · Validate raw files are present

In [ ]:
import os

RAW_PATH = "Files/raw"
EXPECTED = ["seed_courses.json", "seed_learners.json", "seed_xapi_events.json"]

for fname in EXPECTED:
    path = f"{RAW_PATH}/{fname}"
    exists = os.path.exists(f"/lakehouse/default/{path}")
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {path}")

print("\nIf any files are missing, re-run scripts/02_upload_to_fabric.py locally.")

## 1 · Bronze Layer — ingest raw JSON files

In [ ]:
# ── Bronze: xAPI events ──────────────────────────────────────────────────────
df_events_raw = (
    spark.read
         .option("multiline", "true")
         .json("Files/raw/seed_xapi_events.json")
)

print(f"Raw xAPI events: {df_events_raw.count():,} rows")
df_events_raw.printSchema()

In [ ]:
(
    df_events_raw
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_xapi_events")
)
print("✓ bronze_xapi_events written")

In [ ]:
# ── Bronze: courses dimension ─────────────────────────────────────────────────
df_courses = (
    spark.read
         .option("multiline", "true")
         .json("Files/raw/seed_courses.json")
)
print(f"Courses: {df_courses.count()} rows")

(
    df_courses
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dim_courses")
)
print("✓ dim_courses written")

In [ ]:
# ── Bronze: learners dimension ────────────────────────────────────────────────
df_learners = (
    spark.read
         .option("multiline", "true")
         .json("Files/raw/seed_learners.json")
)
print(f"Learners: {df_learners.count()} rows")

(
    df_learners
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dim_learners")
)
print("✓ dim_learners written")

In [ ]:
# Quick sanity check
spark.sql("SELECT verb.display['en-US'] as verb, COUNT(*) as cnt FROM bronze_xapi_events GROUP BY 1 ORDER BY cnt DESC").show()

## 2 · Silver Layer — validate, flatten, enrich

In [ ]:
df_silver = spark.sql("""
    SELECT
        id                                           AS event_id,
        actor.mbox                                   AS learner_email,
        actor.name                                   AS learner_name,
        verb.display['en-US']                        AS verb,
        object.id                                    AS object_id,
        object.definition.name['en-US']              AS object_name,
        result.score.scaled                          AS score,
        result.score.raw                             AS score_raw,
        result.success                               AS quiz_passed,
        result.completion                            AS course_completed,
        result.duration                              AS duration_iso,
        CAST(timestamp AS TIMESTAMP)                 AS ts,
        context.extensions['https://lms.novatech.example.com/ext/course_id']  AS course_id,
        context.extensions['https://lms.novatech.example.com/ext/module_id']  AS module_id,
        context.extensions['https://lms.novatech.example.com/ext/department'] AS department,
        context.extensions['https://lms.novatech.example.com/ext/format']     AS course_format
    FROM bronze_xapi_events
    WHERE actor.mbox IS NOT NULL
      AND verb.display['en-US'] IS NOT NULL
      AND context.extensions['https://lms.novatech.example.com/ext/course_id'] IS NOT NULL
""")

print(f"Silver rows: {df_silver.count():,}")
df_silver.printSchema()

In [ ]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("course_id")
    .saveAsTable("silver_xapi_events")
)
print("✓ silver_xapi_events written (partitioned by course_id)")

In [ ]:
spark.sql("""
    SELECT
        department,
        verb,
        COUNT(*) AS cnt,
        ROUND(AVG(score), 3) AS avg_score
    FROM silver_xapi_events
    GROUP BY department, verb
    ORDER BY department, cnt DESC
""").show(30)

## 3 · Gold Layer — aggregations for analytics

In [ ]:
# ── Gold: course completion summary ──────────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE TABLE gold_course_completion AS
    SELECT
        s.course_id,
        c.title,
        c.department,
        c.format                                                    AS course_format,
        c.skill_level,
        c.duration_hours,
        COUNT(DISTINCT s.learner_email)                              AS enrolled,
        COUNT(DISTINCT CASE WHEN s.verb = 'completed'
                            AND s.course_completed = true
                       THEN s.learner_email END)                     AS completed,
        ROUND(
            COUNT(DISTINCT CASE WHEN s.verb = 'completed'
                                AND s.course_completed = true
                           THEN s.learner_email END)
            * 100.0
            / NULLIF(COUNT(DISTINCT s.learner_email), 0),
        1)                                                           AS completion_rate_pct,
        ROUND(AVG(CASE WHEN s.verb = 'scored' THEN s.score END), 3) AS avg_score,
        ROUND(AVG(CASE WHEN s.verb = 'scored' THEN s.score_raw END), 1) AS avg_score_raw,
        COUNT(CASE WHEN s.verb = 'scored' AND s.score < 0.60
                   THEN 1 END)                                       AS low_score_events
    FROM silver_xapi_events s
    JOIN dim_courses c ON s.course_id = c.course_id
    GROUP BY s.course_id, c.title, c.department, c.format, c.skill_level, c.duration_hours
""")
print("✓ gold_course_completion")
spark.sql("SELECT * FROM gold_course_completion ORDER BY completion_rate_pct DESC LIMIT 10").show()

In [ ]:
# ── Gold: module drop-off funnel ─────────────────────────────────────────────
spark.sql("""
    CREATE OR REPLACE TABLE gold_module_dropoff AS
    SELECT
        course_id,
        module_id,
        COUNT(DISTINCT learner_email)                                           AS learners_reached,
        COUNT(DISTINCT CASE WHEN verb = 'completed' THEN learner_email END)     AS learners_completed,
        ROUND(
            COUNT(DISTINCT CASE WHEN verb = 'completed' THEN learner_email END)
            * 100.0
            / NULLIF(COUNT(DISTINCT learner_email), 0),
        1)                                                                      AS module_completion_pct,
        ROUND(AVG(CASE WHEN verb = 'scored' THEN score END), 3)                 AS avg_quiz_score,
        COUNT(CASE WHEN verb = 'scored' AND score < 0.60 THEN 1 END)           AS low_score_count
    FROM silver_xapi_events
    GROUP BY course_id, module_id
    ORDER BY course_id, module_id
""")
print("✓ gold_module_dropoff")

# Show the drop-off pattern (module 3–4 should show lower completion in ~30% of courses)
spark.sql("""
    SELECT
        module_id,
        ROUND(AVG(module_completion_pct), 1) AS avg_completion_pct,
        ROUND(AVG(avg_quiz_score), 3)        AS avg_quiz_score,
        COUNT(DISTINCT course_id)            AS courses
    FROM gold_module_dropoff
    GROUP BY module_id
    ORDER BY module_id
""").show()

In [ ]:
# ── Gold: department skill proficiency ───────────────────────────────────────
# Learners table has a nested 'skills' array: [{skill_name, proficiency_level}]
spark.sql("""
    CREATE OR REPLACE TABLE gold_department_skills AS
    SELECT
        l.department,
        s.skill_name,
        ROUND(AVG(s.proficiency_level), 2)  AS avg_proficiency,
        3.5                                  AS target_proficiency,
        ROUND(3.5 - AVG(s.proficiency_level), 2) AS skill_gap,
        COUNT(*)                             AS learner_count
    FROM dim_learners l
    LATERAL VIEW EXPLODE(l.skills) AS s
    GROUP BY l.department, s.skill_name
    ORDER BY l.department, skill_gap DESC
""")
print("✓ gold_department_skills")

# Validate Engineering Python/ML gaps
spark.sql("""
    SELECT department, skill_name, avg_proficiency, skill_gap
    FROM gold_department_skills
    WHERE department = 'Engineering'
    ORDER BY skill_gap DESC
""").show()

In [ ]:
# ── Gold: engagement by hour (heatmap data) ───────────────────────────────────
spark.sql("""
    CREATE OR REPLACE TABLE gold_engagement_hourly AS
    SELECT
        course_id,
        module_id,
        HOUR(ts)            AS hour_of_day,
        DAYOFWEEK(ts)       AS day_of_week,
        COUNT(*)            AS event_count,
        COUNT(DISTINCT learner_email) AS unique_learners
    FROM silver_xapi_events
    WHERE verb IN ('interacted', 'progressed', 'launched')
    GROUP BY course_id, module_id, HOUR(ts), DAYOFWEEK(ts)
""")
print("✓ gold_engagement_hourly")

# Validate peak hours at 10 and 14
spark.sql("""
    SELECT
        hour_of_day,
        SUM(event_count)            AS total_events,
        SUM(unique_learners)        AS total_learners
    FROM gold_engagement_hourly
    GROUP BY hour_of_day
    ORDER BY total_events DESC
    LIMIT 10
""").show()

## 4 · Final verification

In [ ]:
tables = [
    "bronze_xapi_events",
    "dim_courses",
    "dim_learners",
    "silver_xapi_events",
    "gold_course_completion",
    "gold_module_dropoff",
    "gold_department_skills",
    "gold_engagement_hourly",
]

print("Table row counts:")
for t in tables:
    cnt = spark.sql(f"SELECT COUNT(*) AS n FROM {t}").collect()[0].n
    print(f"  {t:35s} {cnt:>10,}")

print("\n✓ Medallion tables ready. Connect via Fabric SQL analytics endpoint.")

In [ ]:
# Validate blended vs self-paced completion rate pattern
spark.sql("""
    SELECT
        course_format,
        COUNT(*)                              AS num_courses,
        ROUND(AVG(completion_rate_pct), 1)    AS avg_completion_rate_pct
    FROM gold_course_completion
    WHERE enrolled >= 5
    GROUP BY course_format
    ORDER BY avg_completion_rate_pct DESC
""").show()